# BUSA 310 — Assignment 2: Exploratory Data Analysis of Asset Prices

**Chosen additional tickers:** AAPL, JNJ, XOM, EFA  
**Sectors represented:** Technology, Healthcare, Energy, International Equity  
**Date range:** 2025-09-06 to 2026-09-06

> This notebook is designed for Google Colab. Run from top to bottom. It will create the required tables/charts and print data-driven draft answers for the written questions.


In [ ]:
# 0. Setup
!pip -q install yfinance

import os, math, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/BUSA310/Assignment2'
CODE_DIR = f'{BASE}/Colab_Codes'
RESULTS_DIR = f'{BASE}/Results'
os.makedirs(CODE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

START = '2025-09-06'
END = '2026-09-06'

# Yahoo Finance uses BTC-USD rather than BTCUSD.
tickers_yf = ['SPY','TSLA','GM','BTC-USD','AAPL','JNJ','XOM','EFA']
display_names = {
    'SPY':'SPY',
    'TSLA':'TSLA',
    'GM':'GM',
    'BTC-USD':'BTCUSD',
    'AAPL':'AAPL',
    'JNJ':'JNJ',
    'XOM':'XOM',
    'EFA':'EFA'
}


## Section 1 — Data Collection and Organization

In [ ]:
# Download raw daily data.
# end is exclusive in yfinance, so add one day to include the requested end date if available.
end_plus_one = (pd.Timestamp(END) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

df = yf.download(
    tickers_yf,
    start=START,
    end=end_plus_one,
    auto_adjust=False,
    group_by='column',
    progress=False,
    threads=True
)

print('FIRST 15 ROWS OF RAW DATA')
display(df.head(15))

print('\nRAW DATA INFO')
df.info()

print('\nRAW DATA DESCRIBE')
display(df.describe())


In [ ]:
# Extract closing prices robustly.
if isinstance(df.columns, pd.MultiIndex):
    lvl0 = df.columns.get_level_values(0)
    price_field = 'Adj Close' if 'Adj Close' in lvl0 else 'Close'
    df_close_raw = df[price_field].copy()
else:
    # Fallback if yfinance returns a single-level DataFrame
    price_field = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
    df_close_raw = df[[price_field]].copy()

df_close_raw = df_close_raw.rename(columns=display_names)

# Count available observations BEFORE aligning calendars.
raw_counts = df_close_raw.notna().sum().sort_values(ascending=False)
print('Available observations before alignment:')
display(raw_counts.to_frame('non_missing_days'))

# For cross-asset comparisons, use only dates on which every asset has a price.
# This avoids forward-filling equity prices over weekends and creating artificial 0% returns.
df_close = df_close_raw.dropna(how='any').copy()

print(f'\nAligned df_close shape: {df_close.shape}')
display(df_close.head())

print('\ndf_close.dtypes')
display(df_close.dtypes)


### Written answers — Section 1

**Q1. Trading days and row counts**

Use the automatically printed values below. The key interpretation is that equities/ETFs do not trade on weekends and U.S. exchange holidays, while Bitcoin trades continuously, including weekends and many holidays. The aligned `df_close` intentionally keeps only common dates across all eight assets.

**Q2. Additional ticker choices**

- **AAPL — Apple Inc. — Technology.** Adds a large-cap technology company and gives the analysis exposure to a sector not represented by SPY, TSLA, GM, or Bitcoin alone.
- **JNJ — Johnson & Johnson — Healthcare.** Adds a defensive healthcare company whose business cycle and risk drivers differ from autos, technology, and cryptocurrency.
- **XOM — Exxon Mobil Corp. — Energy.** Adds direct energy-sector exposure and sensitivity to oil/commodity conditions.
- **EFA — iShares MSCI EAFE ETF — International equity.** Adds developed non-U.S. equity exposure, allowing geographic diversification beyond U.S. stocks.

**Q3. Dtypes**

Ideally all closing-price columns are `float64`. A realistic way a column becomes `object` is if the source contains currency symbols, commas, or nonnumeric text such as `"N/A"`. A robust conversion is:

```python
df_close['AAPL'] = pd.to_numeric(df_close['AAPL'], errors='coerce')
```


In [ ]:
# Auto-answer Q1
print('Q1 — raw non-missing observations:')
for t, n in raw_counts.items():
    print(f'{t}: {n}')

same = raw_counts.nunique() == 1
print(f'\nDo all eight tickers have exactly the same raw count? {same}')
print('Reason: U.S. stocks/ETFs do not trade on weekends and exchange holidays; BTCUSD trades 24/7, so its raw count is larger. '
      'Other small differences can also occur because of missing quotes or source-specific observations.')


## Section 2 — Descriptive Statistics and Risk Metrics

In [ ]:
# Required descriptive statistics: min, max, mean, std of closing PRICES
stats_table = pd.DataFrame({
    'Min': df_close.min(),
    'Max': df_close.max(),
    'Mean': df_close.mean(),
    'Std': df_close.std()
}).round(2)

print('PRICE DESCRIPTIVE STATISTICS')
display(stats_table)

# Annual return using the assignment formula
annual_return = (df_close.iloc[-1] - df_close.iloc[0]) / df_close.iloc[0]
annual_return_pct = (annual_return * 100).round(2)

print('ANNUAL RETURN (%)')
display(annual_return_pct.sort_values(ascending=False).to_frame('Annual Return %'))

# Daily returns
daily_returns = df_close.pct_change(fill_method=None).dropna()

# Sharpe
rf_daily = 0.045 / 252
sharpe = ((daily_returns.mean() - rf_daily) / daily_returns.std()) * np.sqrt(252)
sharpe = sharpe.sort_values(ascending=False)

print('ANNUALIZED SHARPE RATIO')
display(sharpe.round(3).to_frame('Sharpe'))

# 30-day rolling std of DAILY returns, shown as %
rolling_30 = daily_returns.rolling(30).std() * 100
print('LAST 10 ROWS: 30-DAY ROLLING DAILY-RETURN STD (%)')
display(rolling_30.tail(10).round(3))

# Required price-level correlation
price_corr = df_close.corr().round(2)
print('PRICE-LEVEL CORRELATION MATRIX')
display(price_corr)


In [ ]:
# Supporting metrics used for interpretation
ann_vol = daily_returns.std() * np.sqrt(252)
return_corr = daily_returns.corr()

# Utility: unique correlation pairs
def unique_corr_pairs(corr):
    pairs = []
    cols = corr.columns
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            pairs.append((cols[i], cols[j], corr.iloc[i,j]))
    return pd.DataFrame(pairs, columns=['Ticker1','Ticker2','Correlation'])

price_pairs = unique_corr_pairs(df_close.corr())
return_pairs = unique_corr_pairs(return_corr)

highest_price_pair = price_pairs.loc[price_pairs['Correlation'].idxmax()]
lowest_price_pair = price_pairs.loc[price_pairs['Correlation'].idxmin()]
lowest_return_pair = return_pairs.loc[return_pairs['Correlation'].idxmin()]

# Peak rolling volatility date for each ticker
peak_roll = {}
for t in rolling_30.columns:
    s = rolling_30[t].dropna()
    peak_roll[t] = (s.idxmax(), s.max())

print('ANNUALIZED RETURN VOLATILITY (%) — SUPPORTING METRIC')
display((ann_vol*100).round(2).sort_values().to_frame('Annualized return vol %'))

print('\nHighest price-level correlation pair:')
print(highest_price_pair.to_dict())
print('Lowest price-level correlation pair:')
print(lowest_price_pair.to_dict())
print('Lowest RETURN-correlation pair (better diversification evidence):')
print(lowest_return_pair.to_dict())

print('\nPeak 30-day rolling daily-return volatility by ticker:')
for t,(d,v) in peak_roll.items():
    print(f'{t}: {d.date()} — {v:.2f}% daily std')


In [ ]:
# Draft answers Q5-Q10 based on actual output
ranking = annual_return_pct.sort_values(ascending=False)
best_ret_ticker = ranking.index[0]
best_ret = ranking.iloc[0]
value_1000 = 1000 * (1 + best_ret/100)

best_sharpe_ticker = sharpe.index[0]
best_sharpe = sharpe.iloc[0]

low_price_std_ticker = stats_table['Std'].idxmin()
low_price_std = stats_table.loc[low_price_std_ticker,'Std']

print('Q5 DRAFT')
print('Return ranking:', ' > '.join([f'{t} ({ranking[t]:.2f}%)' for t in ranking.index]))
print(f'Best return: {best_ret_ticker} at {best_ret:.2f}%.')
print(f'$1,000 × (1 + {best_ret:.4f}) = ${value_1000:,.2f}.\n')

print('Q6 DRAFT')
print(f'Highest Sharpe: {best_sharpe_ticker} ({best_sharpe:.3f}).')
print(f'Highest annual return: {best_ret_ticker} ({best_ret:.2f}%).')
if best_sharpe_ticker == best_ret_ticker:
    print('They are the same asset in this sample.')
else:
    print('They are different. This shows that the asset with the largest raw gain did not necessarily deliver the best return per unit of volatility. '
          'For comparing investments with different risk, Sharpe is generally more informative than raw return, although it should not be used alone.')

print('\nQ7 DATA ANCHOR')
# Choose the ticker with the single highest peak 30-day vol
q7_ticker = max(peak_roll, key=lambda k: peak_roll[k][1])
q7_date, q7_vol = peak_roll[q7_ticker]
print(f'Use {q7_ticker}: peak 30-day daily-return volatility = {q7_vol:.2f}% around {q7_date.date()}.')
print('Do a brief news check around this date and connect the event to the volatility spike; do not claim causality from the chart alone.')

print('\nQ8 DRAFT')
print(f'Highest price-level correlation: {highest_price_pair.Ticker1}-{highest_price_pair.Ticker2} = {highest_price_pair.Correlation:.2f}.')
print(f'Lowest price-level correlation: {lowest_price_pair.Ticker1}-{lowest_price_pair.Ticker2} = {lowest_price_pair.Correlation:.2f}.')
print(f'For diversification, I would focus more on return correlations; the lowest return-correlation pair is '
      f'{lowest_return_pair.Ticker1}-{lowest_return_pair.Ticker2} = {lowest_return_pair.Correlation:.2f}. '
      'A lower return correlation means the two assets tend to move less closely on a day-to-day basis, which can reduce portfolio volatility.')

print('\nQ9 DRAFT')
print('Correlating price levels can be spurious because many asset prices are non-stationary and trend over time. '
      'Two unrelated assets can both trend upward and therefore show a high correlation even when their day-to-day innovations are weakly related. '
      'Return correlations should be computed instead. They are usually lower in magnitude than price-level correlations and better reflect co-movement in investable changes.')

print('\nQ10 DATA ANCHOR')
print(f'Lowest closing-price standard deviation: {low_price_std_ticker} = {low_price_std:.2f}.')
print(f'Its annual return = {annual_return_pct[low_price_std_ticker]:.2f}% and Sharpe = {sharpe[low_price_std_ticker]:.3f}.')
alt = best_sharpe_ticker
print(f'Compare with {alt}: annual return = {annual_return_pct[alt]:.2f}%, Sharpe = {sharpe[alt]:.3f}, '
      f'annualized return volatility = {ann_vol[alt]*100:.2f}%.')
print('A higher-volatility asset can be rational if the additional expected/realized return more than compensates for the extra risk, producing a superior risk-adjusted return. '
      'Also, the standard deviation of PRICE LEVELS is scale-dependent, so it is not a clean cross-asset risk measure.')


## Section 3 — Visualization

In [ ]:
# Part A: Raw price chart
plt.figure(figsize=(14,7))
for c in df_close.columns:
    plt.plot(df_close.index, df_close[c], label=c)
plt.title('Raw Closing Prices — Eight Assets')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend(loc='upper center', bbox_to_anchor=(0.5,-0.12), ncol=4)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/raw_prices.png', dpi=200, bbox_inches='tight')
plt.show()

# Normalized to 100
normalized_df_close = df_close.div(df_close.iloc[0]).mul(100)

plt.figure(figsize=(14,7))
for c in normalized_df_close.columns:
    plt.plot(normalized_df_close.index, normalized_df_close[c], label=c)
plt.title('Normalized Price Index (First Day = 100)')
plt.xlabel('Date')
plt.ylabel('Index Level')
plt.legend(loc='upper center', bbox_to_anchor=(0.5,-0.12), ncol=4)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/normalized_prices.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Q11: objectively find the two normalized lines that are hardest to distinguish
pairs = []
for a,b in itertools.combinations(normalized_df_close.columns,2):
    mean_abs_gap = (normalized_df_close[a]-normalized_df_close[b]).abs().mean()
    pairs.append((a,b,mean_abs_gap))
closest_pair = min(pairs, key=lambda x:x[2])

# Q12: assets outperforming SPY
final_norm = normalized_df_close.iloc[-1]
outperformers = final_norm[final_norm > final_norm['SPY']].sort_values(ascending=False)

# Find a 63-trading-day (~3 month) window ending when cross-sectional performance spread is greatest
spread = normalized_df_close.max(axis=1) - normalized_df_close.min(axis=1)
spread_end = spread.idxmax()
spread_start = normalized_df_close.index[max(0, normalized_df_close.index.get_loc(spread_end)-63)]
best_on_spread_end = normalized_df_close.loc[spread_end].idxmax()
worst_on_spread_end = normalized_df_close.loc[spread_end].idxmin()

print('Q11 DRAFT')
print('The raw chart is poor for comparing performance because assets begin at very different dollar price levels, so high-priced series dominate the vertical scale and smaller-priced series are visually compressed. '
      'Normalizing every asset to 100 on the first day converts the chart from dollar price levels to comparable cumulative performance indices.')
print(f'The hardest pair to distinguish is {closest_pair[0]} and {closest_pair[1]}, with the smallest average normalized-index gap ({closest_pair[2]:.2f} points).\n')

print('Q12 DATA')
print('Final normalized levels:')
display(final_norm.sort_values(ascending=False).round(2).to_frame('Final normalized index'))
print('Outperformed SPY:', list(outperformers.index))
print(f'Largest cross-sectional spread occurs around {spread_end.date()}; inspect roughly {spread_start.date()} to {spread_end.date()}.')
print(f'At the spread end, best = {best_on_spread_end}; worst = {worst_on_spread_end}; spread = {spread.loc[spread_end]:.2f} normalized points.')


In [ ]:
# Part B: Daily return subplots
returns_pct = daily_returns * 100

fig, axes = plt.subplots(4,2, figsize=(15,13), sharex=True)
axes = axes.ravel()
for ax,t in zip(axes, returns_pct.columns):
    ax.plot(returns_pct.index, returns_pct[t], linewidth=0.8)
    ax.axhline(0, linewidth=1)
    ax.set_title(t)
    ax.set_ylabel('Daily Return (%)')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/daily_return_subplots.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Q13: largest absolute daily move
stacked_abs = returns_pct.abs().stack()
largest_idx = stacked_abs.idxmax()
largest_date, largest_ticker = largest_idx
largest_move = returns_pct.loc[largest_date, largest_ticker]

# Same-direction co-movement:
# Find date maximizing count of >2% positive moves or <-2% negative moves.
pos_count = (returns_pct > 2).sum(axis=1)
neg_count = (returns_pct < -2).sum(axis=1)
co_date = pd.concat([pos_count.rename('pos'),neg_count.rename('neg')],axis=1).max(axis=1).idxmax()
direction = 'positive' if pos_count.loc[co_date] >= neg_count.loc[co_date] else 'negative'
count_same = max(pos_count.loc[co_date], neg_count.loc[co_date])

print('Q13 DRAFT')
print(f'Largest single daily move: {largest_ticker} on {largest_date.date()}, {largest_move:.2f}%.')
print(f'A strong same-direction co-movement date is {co_date.date()}: {count_same} assets had moves larger than 2% in the {direction} direction.')
print('Such co-movement suggests exposure to a common market or macro factor rather than purely asset-specific news.')


### Q14 — `.pct_change()` explanation

For a price series \(P_t\),

\[
r_t = \frac{P_t-P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}}-1.
\]

Equivalent arithmetic-only code for one row:

```python
(current_price - previous_price) / previous_price
```

The first row is `NaN` because there is no previous observation against which to calculate a change. If the previous price is zero, the calculation divides by zero and produces an infinite/undefined return; that observation must be investigated or cleaned because a conventional percentage return is not meaningful in that case.


In [ ]:
# Part C: Histograms
fig, axes = plt.subplots(4,2, figsize=(15,13))
axes = axes.ravel()
for ax,t in zip(axes, returns_pct.columns):
    ax.hist(returns_pct[t].dropna(), bins=30, edgecolor='black', alpha=0.75)
    ax.axvline(returns_pct[t].mean(), linewidth=1.5)
    ax.set_title(t)
    ax.set_xlabel('Daily Return (%)')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/return_histograms.png', dpi=200, bbox_inches='tight')
plt.show()

# Shape metrics
skewness = daily_returns.skew()
excess_kurt = daily_returns.kurt()  # pandas returns excess kurtosis

shape_table = pd.DataFrame({
    'Skewness': skewness,
    'Excess Kurtosis': excess_kurt
})

def shape_label(sk, ku):
    if ku > 1:
        base = 'heavy-tailed'
    elif sk > 0.5:
        base = 'right-skewed'
    elif sk < -0.5:
        base = 'left-skewed'
    else:
        base = 'roughly symmetric'
    return base

shape_table['Shape'] = [shape_label(shape_table.loc[t,'Skewness'], shape_table.loc[t,'Excess Kurtosis']) for t in shape_table.index]
shape_table['Normality score'] = shape_table['Skewness'].abs() + shape_table['Excess Kurtosis'].abs()
shape_table = shape_table.sort_values('Normality score')

display(shape_table.round(3))

most_heavy = excess_kurt.idxmax()
print('Q15 DRAFT')
print('Most normal-looking to least (using |skew| + |excess kurtosis|):')
print(' > '.join(shape_table.index))
print(f'Most heavy-tailed by excess kurtosis: {most_heavy}, excess kurtosis = {excess_kurt[most_heavy]:.2f}.')
print('Positive excess kurtosis means the distribution has more probability in the tails than a normal distribution. '
      'Therefore standard deviation alone can understate the practical importance of rare, extreme moves.')


In [ ]:
# Part D: Required PRICE-LEVEL correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df_close.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap of Closing Price Levels')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/price_correlation_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

# Q16: simple cluster summary based on average price correlation
avg_corr = df_close.corr().apply(lambda s: (s.sum()-1)/(len(s)-1))
most_independent = avg_corr.idxmin()

# strongest 3-asset cluster: maximize mean pairwise corr
best_cluster = None
best_cluster_corr = -999
for combo in itertools.combinations(df_close.columns,3):
    sub = df_close[list(combo)].corr()
    vals = [sub.iloc[0,1], sub.iloc[0,2], sub.iloc[1,2]]
    m = np.mean(vals)
    if m > best_cluster_corr:
        best_cluster_corr = m
        best_cluster = combo

print('Q16 DRAFT')
print(f'Strongest 3-asset cluster by average price-level correlation: {best_cluster}, mean pairwise correlation = {best_cluster_corr:.2f}.')
print(f'Most independent by lowest average correlation with the rest: {most_independent}, average correlation = {avg_corr[most_independent]:.2f}.')
print('A relatively independent asset can improve diversification because its movements are less synchronized with the rest of the portfolio; '
      'however, return correlations are more defensible than price-level correlations for this conclusion.')


## Section 4 — Portfolio Construction

Client: David, age 61, retirement in 6 years, $40,000 investable, maximum tolerable one-year loss 25%, wants returns meaningfully above 4.5% with limited downside.

The allocation below deliberately emphasizes diversified/defensive assets and assigns 0% to the most speculative positions.


In [ ]:
# Q17 — conservative allocation
allocation = pd.Series({
    'SPY': 45,
    'TSLA': 0,
    'GM': 0,
    'BTCUSD': 0,
    'AAPL': 5,
    'JNJ': 20,
    'XOM': 15,
    'EFA': 15
}, dtype=float)

assert abs(allocation.sum()-100) < 1e-9
display(allocation.to_frame('Allocation %'))

dollars = allocation/100 * 40000
display(dollars.to_frame('Dollar Allocation'))

print('Q17 DRAFT')
print('SPY 45%, AAPL 5%, JNJ 20%, XOM 15%, EFA 15%, TSLA 0%, GM 0%, BTCUSD 0%.')
print('The portfolio emphasizes broad-market and defensive diversification. TSLA and BTCUSD are excluded because the client explicitly rejects speculative positions; '
      'GM is excluded to avoid adding another concentrated cyclical auto position when SPY already contains diversified U.S. equity exposure.')


In [ ]:
# Q18 — print one quantitative anchor for each non-zero holding
print('Q18 — QUANTITATIVE ANCHORS')
for t,w in allocation[allocation>0].items():
    print(f'{t} ({w:.0f}%): annual return {annual_return_pct[t]:.2f}%, Sharpe {sharpe[t]:.3f}, '
          f'annualized return volatility {ann_vol[t]*100:.2f}%, avg return correlation with others '
          f'{((return_corr.loc[t].sum()-1)/(len(return_corr)-1)):.2f}.')


### Q18 — How to write the justification

Use the printed figures above and make the **single most important quantitative reason** explicit for each non-zero holding. A strong final structure is:

- **SPY (45%)** — core holding because its Sharpe ratio of **[printed value]** and annualized volatility of **[printed value]%** provide a stronger risk/return balance than the more speculative assets.
- **JNJ (20%)** — included because **[choose its strongest printed statistic: lower volatility / lower average return correlation / Sharpe]** supports a defensive role.
- **XOM (15%)** — included because **[printed return/Sharpe/correlation]** adds an energy exposure with different return drivers from technology and autos.
- **EFA (15%)** — included because its **[printed average return correlation]** with the rest of the portfolio provides geographic diversification.
- **AAPL (5%)** — retained only as a small growth sleeve because **[printed return and Sharpe]** show upside potential, while the limited 5% weight controls concentration risk.

For 0% positions, explain the client-fit reason rather than inventing a statistic.


In [ ]:
# Q19 — Stress test
# Assignment says all equity markets drop 20% uniformly.
# This allocation is 100% in equity/ETF positions, so portfolio falls approximately 20%.
initial_value = 40000
stress_loss_pct = 0.20
stress_value = initial_value * (1-stress_loss_pct)
stress_loss_dollars = initial_value - stress_value

print('Q19 DRAFT')
print(f'Initial portfolio = ${initial_value:,.2f}')
print(f'20% stress loss = ${initial_value:,.2f} × 0.20 = ${stress_loss_dollars:,.2f}')
print(f'Ending value = ${initial_value:,.2f} − ${stress_loss_dollars:,.2f} = ${stress_value:,.2f}')
print(f'Loss percentage = 20.00%, which is below David\'s 25% maximum.')
print('Therefore the specified 20% uniform-equity stress scenario does not violate the stated constraint, so no revision is required under this particular stress test.')


### Q20 — Limitations

Three specific limitations:

1. **One year is too short for a six-year horizon.** It may capture a single market regime and miss recessions, rate cycles, crashes, or sector rotations that could materially alter expected returns and risk.
2. **Pearson correlations on price levels can be spurious.** Trending, non-stationary prices can appear highly correlated even when day-to-day returns are not. Return correlations, ideally across multiple regimes, are more appropriate.
3. **Historical sample estimates are unstable and backward-looking.** One-year returns, Sharpe ratios, and volatility can change sharply and do not guarantee future performance.

One client fact that would materially change the allocation is **David's other retirement assets and guaranteed income**. If he already has substantial pension/Social Security income and a large bond portfolio, he may be able to tolerate more equity risk here; if this $40,000 is most of his liquid retirement savings, the allocation should be substantially more defensive.


## Section 5 — Additional Analysis

In [ ]:
# Additional Analysis 1: Maximum drawdown
wealth_index = (1 + daily_returns).cumprod()
running_peak = wealth_index.cummax()
drawdown = wealth_index / running_peak - 1
max_drawdown = drawdown.min().sort_values()

print('ADDITIONAL ANALYSIS 1 — MAXIMUM DRAWDOWN')
display((max_drawdown*100).round(2).to_frame('Max Drawdown %'))

plt.figure(figsize=(14,7))
for t in drawdown.columns:
    plt.plot(drawdown.index, drawdown[t]*100, label=t)
plt.title('Drawdown from Prior Peak')
plt.xlabel('Date')
plt.ylabel('Drawdown (%)')
plt.legend(loc='upper center', bbox_to_anchor=(0.5,-0.12), ncol=4)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/max_drawdown.png', dpi=200, bbox_inches='tight')
plt.show()


**Additional Analysis 1 — Description:**  
I computed each asset's maximum drawdown, which measures the largest peak-to-trough percentage loss during the sample. Unlike standard deviation, drawdown focuses directly on the downside experience an investor would have faced, making it especially relevant for a client who has an explicit loss constraint.


In [ ]:
# Additional Analysis 2: RETURN correlation heatmap
plt.figure(figsize=(10,8))
sns.heatmap(return_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap of Daily Returns')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/return_correlation_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

print('ADDITIONAL ANALYSIS 2 — RETURN CORRELATIONS')
display(return_corr.round(2))


**Additional Analysis 2 — Description:**  
I recomputed correlations using daily returns rather than price levels. This removes much of the common trending behavior that can create spurious price-level correlations and gives a more meaningful view of whether assets actually move together from day to day, which is the relationship that matters for diversification.


## Bonus — Investment Memo (300–400 words)

After running the notebook, replace the bracketed values with the printed outputs.

**To: David and Spouse  
From: Junior Analyst  
Subject: Six-Year Investment Allocation**

I analyzed one year of daily closing prices from **September 6, 2025 through September 6, 2026** for eight assets: SPY, TSLA, GM, BTCUSD, AAPL, JNJ, XOM, and EFA. SPY serves as the broad U.S. equity benchmark, while TSLA and GM provide auto exposure and BTCUSD represents cryptocurrency. I added AAPL for technology, JNJ for healthcare, XOM for energy, and EFA for developed international equities so the dataset would contain multiple sectors and geographic exposures.

Three findings were most important for the recommendation. First, **[best-return ticker]** produced the highest one-year return at **[x.xx%]**, while the highest Sharpe ratio was **[ticker, x.xxx]**, showing that raw return and risk-adjusted return were not necessarily the same. Second, **[lowest-volatility or defensive ticker]** had annualized return volatility of **[x.xx%]**, compared with **[x.xx%]** for **[higher-risk ticker]**. Third, the daily-return correlation between **[diversifying pair]** was **[x.xx]**, indicating meaningfully different day-to-day behavior and supporting diversification.

Given your six-year retirement horizon and stated inability to tolerate a one-year loss greater than 25%, I recommend **45% SPY, 20% JNJ, 15% XOM, 15% EFA, and 5% AAPL**, with **0% in TSLA, GM, and BTCUSD**. SPY is the core diversified holding, while JNJ, XOM, and EFA broaden sector and geographic exposure. AAPL is limited to 5% so the portfolio retains some growth exposure without becoming concentrated. The excluded positions do not fit your preference for steady growth and limited speculation.

Under the required stress test in which equity markets fall uniformly by 20%, the $40,000 portfolio would decline by **$8,000** to approximately **$32,000**, a **20% loss**, which remains inside your 25% limit. The biggest weaknesses are that the analysis uses only one year of historical data and that price-level Pearson correlations can be distorted by common trends. I would also want to know the size and composition of your other retirement assets before making a final real-world recommendation.


## Appendix A — AI Interaction Log

Record this interaction honestly. Example row:

| # | Tool | Prompt summary | Section | What you changed or verified |
|---|---|---|---|---|
| 1 | ChatGPT | Helped build/debug the Assignment 2 Colab workflow, calculations, charts, and explanations | 1–6 | Ran all code, checked outputs against formulas, verified ticker/date choices, and edited final written responses to match the observed results |
